# Architecture Comparison: 5-Step Contradiction-Engine Pipeline vs. Single-Verifier Pipeline

Compares two versions of your AMG-RAG system side by side on the same labeled test
set, using **Accuracy, Precision, Recall, F1, AUROC, and latency**.

| | **System A** | **System B** |
|---|---|---|
| Class name | `AMG_RAG_ReportSystem` | `AMG_RAG_System` |
| LLM backend | Google Gemini/Gemma (`gemma-4-31b-it`) | Google Gemini/Gemma (`gemma-4-31b-it`) -- same `GOOGLE_API_KEY` and same model as System A, so the comparison isolates pipeline architecture rather than LLM choice |
| Finding extraction | Yes | Yes |
| Baseline typical/atypical classification | **Yes** -- a dedicated step decides which findings need literature grounding | **No** -- every finding goes through the same single verifier regardless |
| Literature grounding | PubMed, gated to atypical findings only | PubMed/Wikipedia context, always run on a generic anatomical-target query (not gated) |
| Dedicated contradiction engine | **Yes** -- separate step explicitly checks same-structure conflicts, self-contradictions, and anatomically impossible claims, with `[index, index]` encoding for single-finding contradictions | **No** -- contradiction/hallucination judgment is folded into one monolithic per-finding "grounding score + hallucination indicator" call |
| Vector store | No (flat KG context string) | Yes -- Chroma + HuggingFace sentence-transformer embeddings (declared in `__init__`, not used inside `evaluate_medical_report` itself) |

If "System A" and "System B" map onto "previous"/"current" backwards from how you think
of them, just swap the labels when you read the results -- the metrics themselves don't
depend on which name you call which.

> ⚠️ **Security note:** the notebook you uploaded had two live API keys (Groq, Cerebras)
> hardcoded in a cell. **Rotate both immediately** if you haven't already -- neither key
> is reproduced anywhere in this notebook; both are entered securely via `getpass` instead.

> ⚠️ **Dependency conflict risk:** System A pins `langchain-classic` (newer langchain's
> restructured namespace) while System B pins `langchain==0.3.27` (the pre-restructuring
> namespace) alongside `langchain-chroma`/`langchain-huggingface`/`langchain-openai`.
> Installing both in one live kernel session risks real version conflicts. This notebook
> runs each system in its **own Part**, saving results to disk -- **restart the runtime
> between Part 1 and Part 2** (Runtime -> Restart runtime) rather than trying to run both
> back to back in the same session. Part 3 then just loads both saved result files and
> builds the comparison -- it doesn't need either system's live objects.

## Shared labeled test set

Both parts use the exact same 23-report labeled set (defined identically in each Part
below, so each Part is fully self-contained and re-runnable after a runtime restart).

---
# Part 1: System A (5-Step Contradiction-Engine Pipeline)

Run this part completely, then **restart the runtime** before starting Part 2.

### 1.1 Install dependencies

In [ ]:
%pip install -q "google-auth==2.49.0" "langchain-google-genai" "langchain-classic" networkx requests python-decouple pandas tqdm scikit-learn


### 1.2 API key

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")

pubmed_key = getpass("Enter your PubMed API key (optional, press Enter to skip): ")
if pubmed_key:
    os.environ["pubmed_api"] = pubmed_key

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
print("API key configured.")


### 1.3 System A source

In [ ]:
"""
AMG-RAG: Clinical Report Hallucination Detection System
5-Step Pipeline:
  1. Report Ingestion & Extraction
  2. LLM Baseline Check (typical / atypical)
  3. PubMed Grounding — only if atypical
  4. MKG Contradiction Engine
  5. Final Verdict: Hallucinated / Not Hallucinated with confidence score
"""

import time
import os
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field
import networkx as nx
from langchain_google_genai import ChatGoogleGenerativeAI  # pyright: ignore[reportMissingImports]
from langchain_core.prompts import PromptTemplate  # pyright: ignore[reportMissingImports]
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser  # pyright: ignore[reportMissingImports]
import requests
from xml.etree import ElementTree as ET
from decouple import config  # pyright: ignore[reportMissingImports]

# Configuration
GOOGLE_API_KEY = config('GOOGLE_API_KEY', default=os.environ.get('GOOGLE_API_KEY'))
PUBMED_API_KEY = config('pubmed_api', default=os.environ.get('pubmed_api') or None)


# ──────────────────────────────────────────────
# Data Classes
# ──────────────────────────────────────────────
@dataclass
class MedicalEntity:
    """Represents a medical entity in the knowledge graph"""
    name: str
    description: str
    entity_type: str
    confidence: float = 1.0
    sources: List[str] = field(default_factory=list)

@dataclass
class MedicalRelation:
    """Represents a relationship between medical entities"""
    source: str
    target: str
    relation_type: str
    confidence: float
    evidence: str
    sources: List[str] = field(default_factory=list)


# ──────────────────────────────────────────────
# Medical Knowledge Graph
# ──────────────────────────────────────────────
class MedicalKnowledgeGraph:
    """Dynamic Medical Knowledge Graph with confidence scoring"""

    def __init__(self):
        self.graph = nx.DiGraph()
        self.entities = {}
        self.relations = []

    def add_entity(self, entity: MedicalEntity):
        self.entities[entity.name] = entity
        self.graph.add_node(
            entity.name,
            description=entity.description,
            entity_type=entity.entity_type,
            confidence=entity.confidence,
            sources=entity.sources,
        )

    def add_relation(self, relation: MedicalRelation):
        self.relations.append(relation)
        self.graph.add_edge(
            relation.source,
            relation.target,
            relation_type=relation.relation_type,
            confidence=relation.confidence,
            evidence=relation.evidence,
            sources=relation.sources,
        )

    def get_connected_nodes(self, node_name: str, confidence_threshold: float = 0.0):
        connected = []
        if node_name in self.graph:
            for neighbor in self.graph.neighbors(node_name):
                edge_data = self.graph[node_name][neighbor]
                if edge_data.get("confidence", 0) >= confidence_threshold:
                    connected.append(
                        {
                            "node": neighbor,
                            "relation": edge_data.get("relation_type"),
                            "confidence": edge_data.get("confidence"),
                            "evidence": edge_data.get("evidence"),
                        }
                    )
        return connected


# ──────────────────────────────────────────────
# PubMed Searcher
# ──────────────────────────────────────────────
class PubMedSearcher:
    """PubMed API wrapper for medical literature search"""

    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key
        self.base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    def search(self, query: str, max_results: int = 3) -> List[str]:
        time.sleep(0.5)
        search_url = f"{self.base_url}/esearch.fcgi"
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmode": "xml",
            "retmax": max_results,
        }
        if self.api_key:
            search_params["api_key"] = self.api_key

        try:
            response = requests.get(search_url, params=search_params, timeout=30)
            if response.status_code != 200:
                return []
            if not response.text.strip().startswith("<"):
                return []

            root = ET.fromstring(response.text)
            pmids = [id_elem.text for id_elem in root.findall(".//Id")]
            if not pmids:
                return []

            time.sleep(0.5)
            fetch_url = f"{self.base_url}/efetch.fcgi"
            fetch_params = {
                "db": "pubmed",
                "id": ",".join(pmids),
                "retmode": "text",
                "rettype": "abstract",
            }
            if self.api_key:
                fetch_params["api_key"] = self.api_key

            response = requests.get(fetch_url, params=fetch_params, timeout=30)
            if response.status_code != 200:
                return []

            articles = response.text.split("\n\n")
            abstracts = []
            for article in articles:
                lines = article.split("\n")
                abstract_lines = [
                    line
                    for line in lines
                    if line.strip()
                    and not any(
                        skip in line.lower()
                        for skip in ["author", "doi", "pmid", "copyright"]
                    )
                ]
                if abstract_lines:
                    abstracts.append(" ".join(abstract_lines))
            return abstracts

        except Exception as e:
            print(f"  PubMed search error: {e}")
            return []


# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────
def _align_lists(*lists, fill="N/A"):
    """Force all extracted arrays to the same length so index-based
    alignment (findings[i] <-> anatomical_targets[i] <-> ...) can't
    silently drift when the LLM returns mismatched array lengths."""
    max_len = max((len(l) for l in lists), default=0)
    return [list(l) + [fill] * (max_len - len(l)) for l in lists]


def _normalize_contradiction_pairs(raw_pairs, num_findings):
    """The LLM is asked for pairs like [0, 2] but the scoring logic needs
    a list of index-lists, e.g. [[0, 2], [1, 3]]. Handle both shapes and
    drop anything that doesn't parse as a valid pair of in-range indices."""
    normalized = []
    if not raw_pairs:
        return normalized

    # Case: flat list like [0, 2, 1, 3] -> treat as consecutive pairs
    if raw_pairs and all(isinstance(x, (int, float)) for x in raw_pairs):
        it = iter(raw_pairs)
        raw_pairs = [[a, b] for a, b in zip(it, it)]

    for pair in raw_pairs:
        if isinstance(pair, (list, tuple)) and len(pair) >= 2:
            try:
                a, b = int(pair[0]), int(pair[1])
            except (TypeError, ValueError):
                continue
            if 0 <= a < num_findings and 0 <= b < num_findings:
                normalized.append([a, b])
    return normalized


# ──────────────────────────────────────────────
# Main System
# ──────────────────────────────────────────────
class AMG_RAG_ReportSystem:
    """AMG-RAG system for Clinical Report Hallucination Detection"""

    def __init__(self, google_api_key: str = None):
        if google_api_key:
            self.llm = ChatGoogleGenerativeAI(
                model="gemma-4-31b-it",
                temperature=0.0,
                google_api_key=google_api_key,
            )
        else:
            raise ValueError("Google API key is required.")

        self.kg = MedicalKnowledgeGraph()
        self.pubmed = PubMedSearcher(api_key=PUBMED_API_KEY)
        self._setup_chains()

    # ──────────────────────────────────────────
    # Chain Setup
    # ──────────────────────────────────────────
    def _setup_chains(self):
        """Setup LLM chains for the 5-step report pipeline"""

        # ── Step 1 chain: Report Ingestion & Extraction ──
        finding_schemas = [
            ResponseSchema(
                name="findings",
                description="List of clinical claims/findings parsed from the report",
                type="array",
            ),
            ResponseSchema(
                name="anatomical_targets",
                description="The anatomical structures associated with each finding",
                type="array",
            ),
            ResponseSchema(
                name="clinical_status",
                description="Clinical observation status for each finding (e.g. 'normal', 'abnormal', 'clear')",
                type="array",
            ),
        ]
        finding_parser = StructuredOutputParser.from_response_schemas(finding_schemas)

        self.finding_extractor = (
            PromptTemplate(
                template="""Extract all individual medical findings/claims and associated anatomical structures from this clinical report.

            Report: {report}

            For each finding/claim, provide:
            1. The finding statement
            2. The anatomical target structure
            3. The clinical status (e.g. normal, abnormal, clear, consolidated, etc.)

            IMPORTANT: The three output arrays (findings, anatomical_targets, clinical_status)
            MUST all be the same length and index-aligned — element i of each array must
            describe the same finding.

            Return in JSON format:
            {format_instructions}""",
                input_variables=["report"],
                partial_variables={
                    "format_instructions": finding_parser.get_format_instructions()
                },
            )
            | self.llm
            | finding_parser
        )

        # ── Step 2 chain: LLM Baseline Check ──
        baseline_schemas = [
            ResponseSchema(
                name="classifications",
                description="For each finding: 'typical' if a standard/expected clinical observation, 'atypical' if unusual, rare, or potentially suspicious",
                type="array",
            ),
            ResponseSchema(
                name="reasoning",
                description="Brief reasoning for each classification",
                type="array",
            ),
        ]
        baseline_parser = StructuredOutputParser.from_response_schemas(
            baseline_schemas
        )

        self.baseline_checker = (
            PromptTemplate(
                template="""You are a clinical radiology expert. For each finding below, classify it as either 'typical' or 'atypical'.

            DEFINITIONS:
            - 'typical': A standard, commonly seen clinical observation (e.g. 'heart is normal size', 'lungs are clear', 'no pleural effusion', 'mild cardiomegaly', 'pneumothorax').
            - 'atypical': A finding that is extremely rare, highly unusual, internally contradictory, physiologically impossible, or highly suspicious. This includes:
              a) Extremely rare congenital conditions (e.g. 'dextrocardia', 'situs inversus')
              b) Internally contradictory statements (e.g. 'normal size with severe cardiomegaly', 'clear lungs with massive consolidation')
              c) Impossible whole-organ counts (e.g. 'patient has 3 lungs')
              d) Impossible or incorrect counts of normal anatomical SUBSTRUCTURES — lobes, chambers,
                 valves, vessels, etc. — relative to known human anatomy. For example: the left lung
                 normally has 2 lobes and the right lung has 3 lobes, so "three lobes in the left lung"
                 is atypical/anatomically impossible. The heart normally has 4 chambers, so "five
                 chambers" would be atypical. Apply this same substructure-count check to any
                 anatomical structure mentioned, not just the examples given here.

            NOTE: Common abnormal findings (e.g. pleural effusion, tumors) are still 'typical'. Extremely rare congenital conditions, impossible/contradictory statements, and impossible substructure counts MUST be flagged as 'atypical'.

            Findings to classify:
            {findings_list}

            IMPORTANT: Return exactly one classification and one reasoning entry per finding listed above, in the same order.

            Return in JSON format:
            {format_instructions}""",
                input_variables=["findings_list"],
                partial_variables={
                    "format_instructions": baseline_parser.get_format_instructions()
                },
            )
            | self.llm
            | baseline_parser
        )

        # ── Step 4 chain: MKG Contradiction Engine ──
        contradiction_schemas = [
            ResponseSchema(
                name="contradictions",
                description="List of contradiction descriptions found between findings. Empty list if none found.",
                type="array",
            ),
            ResponseSchema(
                name="contradiction_pairs",
                description="List of [index_a, index_b] pairs indicating which findings are involved in each contradiction. For a contradiction between two different findings, use their two distinct indices, e.g. [0, 2]. For a contradiction involving only ONE finding (self-contradictory statement, or physiologically/anatomically impossible claim), use the SAME index twice, e.g. [5, 5]. Each element must be a 2-item list. Empty list if none.",
                type="array",
            ),
            ResponseSchema(
                name="severity_scores",
                description="Severity score (0.0 to 1.0) for each contradiction, same order/length as contradictions. 1.0 = definite contradiction, 0.5 = possible, 0.0 = no contradiction.",
                type="array",
            ),
        ]
        contradiction_parser = StructuredOutputParser.from_response_schemas(
            contradiction_schemas
        )

        self.contradiction_detector = (
            PromptTemplate(
                template="""You are a clinical contradiction detection engine. Analyze the following clinical findings from a single medical report for internal contradictions.

            A CONTRADICTION occurs when:
            1. Two DIFFERENT findings about the SAME anatomical structure make opposite claims (e.g. 'heart is normal size' AND 'severe cardiomegaly'). Encode as [index_a, index_b] with two distinct indices.
            2. A single finding contains self-contradictory statements (e.g. 'clear lungs with massive consolidation'). Encode as [index, index] — the SAME index twice, since only one finding is involved.
            3. A finding describes something physiologically or anatomically impossible (e.g. 'patient has 3 lungs', 'three lobes in the left lung', 'five cardiac chambers'). Encode as [index, index] — the SAME index twice, since only one finding is involved.

            NOT a contradiction (these are normal, expected patterns in clinical radiology):
            1. Multiple abnormal findings in different regions (e.g. effusion in lungs + cardiomegaly in heart)
            2. Worsening or progression of a condition across serial reports
            3. Minor/trace findings coexisting with general "clear" or "no acute findings" statements
               - In radiology convention, "clear lungs" means no consolidation/infiltrate/pneumonia, NOT absence of ALL findings
               - Minor findings like mild basilar atelectasis, trace granulomas, or small nodules can coexist with "clear" statements
               - This is standard practice and NOT a contradiction (e.g. "lungs are clear" + "mild basilar atelectasis" = NORMAL, not contradictory)
            4. Incidental findings mentioned alongside primary pathology (e.g. "no pneumonia" + "small granuloma noted")
            5. ACUITY MISMATCH RULE: "No acute findings" is a claim about the absence of new/emergent
                  pathology — it is NOT a claim that the study is entirely normal, and it does NOT
                  contradict a finding unless that finding is itself explicitly framed as acute, new,
                  or an interval change.

                  To evaluate whether a pairing is a contradiction, classify the abnormal finding's
                  acuity based on its own wording:
                  - Explicitly acute/new/interval-change language ("new," "acute," "increased from prior,"
                    "worsening," "interval development of") → CAN contradict "no acute findings"
                  - No acuity language, or language suggesting a chronic/structural/incidental process
                    ("enlarged," "prominent," "tortuous," "calcified," "stable," "chronic," "old,"
                    "longstanding") → does NOT contradict "no acute findings" on its own

                  Do not infer acuity from the finding's anatomical category or clinical severity —
                  infer it only from the acuity language actually present in that finding's text.
                  If acuity is ambiguous or unstated, do NOT flag it as a contradiction
                  (default to non-contradiction when acuity is unclear).

                  Examples (illustrative only, not exhaustive): enlarged pulmonary arteries, old rib
                  fracture, stable pulmonary nodule, chronic scarring, calcified granuloma.

            Findings:
            {findings_list}

            Knowledge Graph Context:
            {kg_context}

            Identify ONLY true logical contradictions. Each entry in contradiction_pairs MUST be
            a 2-item list of 0-based finding indices — [index_a, index_b] for a two-finding
            contradiction (type 1), or [index, index] for a single-finding contradiction
            (types 2 and 3). contradictions / contradiction_pairs / severity_scores MUST all be
            the same length, in matching order.

            Return in JSON format:
            {format_instructions}""",
                input_variables=["findings_list", "kg_context"],
                partial_variables={
                    "format_instructions": contradiction_parser.get_format_instructions()
                },
            )
            | self.llm
            | contradiction_parser
        )

    # ──────────────────────────────────────────
    # Main Pipeline
    # ──────────────────────────────────────────
    def evaluate_medical_report(self, report: str) -> Dict[str, Any]:
        """Execute the 5-step report hallucination detection pipeline."""

        # ════════════════════════════════════════
        # STEP 1: Report Ingestion & Extraction
        # ════════════════════════════════════════
        print("\n" + "=" * 60)
        print("  Step 1: Report Ingestion & Extraction")
        print("=" * 60)
        print(f"\n  Input Report:\n  \"{report}\"\n")

        try:
            finding_result = self.finding_extractor.invoke({"report": report})
            findings = finding_result.get("findings", []) or []
            anatomical_targets = finding_result.get("anatomical_targets", []) or []
            clinical_statuses = finding_result.get("clinical_status", []) or []
        except Exception as e:
            print(f"  Finding extraction error: {e}")
            findings = [s.strip() for s in report.split(".") if s.strip()]
            anatomical_targets = ["General"] * len(findings)
            clinical_statuses = ["Unspecified"] * len(findings)

        # FIXED: enforce index alignment instead of relying on ad-hoc
        # "if i < len(...)" guards scattered through the rest of the pipeline.
        findings, anatomical_targets, clinical_statuses = _align_lists(
            findings, anatomical_targets, clinical_statuses
        )

        print(f"  Extracted {len(findings)} clinical claims:\n")
        for i, f in enumerate(findings):
            print(f"    Claim {i+1}: \"{f}\"")
            print(f"             Target: {anatomical_targets[i]} | Status: {clinical_statuses[i]}")

        # ════════════════════════════════════════
        # STEP 2: LLM Baseline Check
        # ════════════════════════════════════════
        print("\n" + "=" * 60)
        print("  Step 2: LLM Baseline Check (typical / atypical)")
        print("=" * 60)

        findings_formatted = "\n".join(
            [
                f"  Finding {i+1}: \"{findings[i]}\" (Target: {anatomical_targets[i]}, Status: {clinical_statuses[i]})"
                for i in range(len(findings))
            ]
        )

        try:
            baseline_result = self.baseline_checker.invoke(
                {"findings_list": findings_formatted}
            )
            classifications = baseline_result.get("classifications", []) or []
            baseline_reasoning = baseline_result.get("reasoning", []) or []
        except Exception as e:
            print(f"  Baseline check error: {e}")
            classifications = []
            baseline_reasoning = []

        # FIXED: pad/align instead of silently defaulting everything to
        # "typical" on a partial or failed response.
        classifications, baseline_reasoning = _align_lists(
            classifications, baseline_reasoning, fill="typical"
        )

        atypical_indices = []
        for i in range(len(findings)):
            cls = classifications[i]
            reason = baseline_reasoning[i]
            marker = "[!] ATYPICAL" if cls.lower() == "atypical" else "[OK] TYPICAL"
            print(f"\n    Claim {i+1}: {marker}")
            print(f"      \"{findings[i]}\"")
            print(f"      Reasoning: {reason}")
            if cls.lower() == "atypical":
                atypical_indices.append(i)

        print(f"\n  Summary: {len(atypical_indices)} atypical / {len(findings)} total findings")

        # ════════════════════════════════════════
        # STEP 3: PubMed Grounding (only if atypical)
        # ════════════════════════════════════════
        print("\n" + "=" * 60)
        print("  Step 3: PubMed Grounding — only if atypical")
        print("=" * 60)

        pubmed_evidence = {}  # index -> list of abstracts
        if not atypical_indices:
            print("\n  No atypical findings detected. Skipping PubMed grounding.")
        else:
            for idx in atypical_indices:
                finding_text = findings[idx]
                target = anatomical_targets[idx]
                query = f"{target} {finding_text}"
                print(f"\n    Searching PubMed for Claim {idx+1}: \"{finding_text[:80]}...\"")
                abstracts = self.pubmed.search(query, max_results=2)
                pubmed_evidence[idx] = abstracts
                if abstracts:
                    print(f"      Found {len(abstracts)} relevant article(s)")
                    for j, abstract in enumerate(abstracts):
                        print(f"        Article {j+1}: {abstract[:120]}...")
                else:
                    print("      No PubMed articles found — finding remains ungrounded")

        # ════════════════════════════════════════
        # STEP 4: MKG Contradiction Engine
        # ════════════════════════════════════════
        print("\n" + "=" * 60)
        print("  Step 4: MKG Contradiction Engine")
        print("=" * 60)

        # Build Knowledge Graph from findings
        self.kg = MedicalKnowledgeGraph()

        print("\n  Building Medical Knowledge Graph from report claims...")
        for i, target in enumerate(anatomical_targets):
            entity_name = target.title()
            if entity_name not in self.kg.entities:
                med_entity = MedicalEntity(
                    name=entity_name,
                    description=f"Anatomical region: {entity_name}. Status: {clinical_statuses[i]}",
                    entity_type="anatomical_structure",
                    confidence=1.0,
                    sources=["Report Parser"],
                )
                self.kg.add_entity(med_entity)

            finding_name = f"Claim_{i+1}"
            finding_entity = MedicalEntity(
                name=finding_name,
                description=findings[i],
                entity_type="clinical_finding",
                confidence=1.0,
                sources=["Report Parser"],
            )
            self.kg.add_entity(finding_entity)

            relation = MedicalRelation(
                source=entity_name,
                target=finding_name,
                relation_type="has_finding",
                confidence=1.0,
                evidence=f"Status: {clinical_statuses[i]}",
                sources=["Report Parser"],
            )
            self.kg.add_relation(relation)

        print(f"    Nodes: {len(self.kg.entities)} | Edges: {len(self.kg.relations)}")

        # Print KG structure
        for entity_name in self.kg.entities.keys():
            connections = self.kg.get_connected_nodes(entity_name)
            if connections:
                for conn in connections:
                    print(f"    {entity_name} --[{conn['relation']}]--> {conn['node']}")

        # Run contradiction detection
        kg_context_lines = []
        for entity_name in self.kg.entities.keys():
            connections = self.kg.get_connected_nodes(entity_name)
            if connections:
                for conn in connections:
                    kg_context_lines.append(
                        f"{entity_name} --[{conn['relation']}]--> {conn['node']}: {self.kg.entities.get(conn['node'], MedicalEntity('','','',0,[])).description}"
                    )
        kg_context = "\n".join(kg_context_lines) if kg_context_lines else "No graph context available."

        print("\n  Running contradiction detection...")
        try:
            contradiction_result = self.contradiction_detector.invoke(
                {"findings_list": findings_formatted, "kg_context": kg_context}
            )
            contradictions = contradiction_result.get("contradictions", []) or []
            raw_pairs = contradiction_result.get("contradiction_pairs", []) or []
            severity_scores = contradiction_result.get("severity_scores", []) or []
        except Exception as e:
            print(f"  Contradiction detection error: {e}")
            contradictions = []
            raw_pairs = []
            severity_scores = []

        # FIXED: robustly normalize pair format instead of assuming the LLM
        # always returns a list of 2-item lists.
        contradiction_pairs = _normalize_contradiction_pairs(raw_pairs, len(findings))

        # Keep contradictions/severity_scores aligned to the (possibly
        # shrunk-by-validation) contradiction_pairs list.
        n = min(len(contradictions), len(contradiction_pairs)) if contradiction_pairs else 0
        contradictions = contradictions[:n]
        contradiction_pairs = contradiction_pairs[:n]
        severity_scores = (severity_scores[:n] if severity_scores else []) + [0.5] * max(0, n - len(severity_scores))

        if contradictions:
            print(f"\n  [!] {len(contradictions)} contradiction(s) detected:\n")
            for i, contradiction in enumerate(contradictions):
                pair = contradiction_pairs[i]
                severity = severity_scores[i]
                print(f"    Contradiction {i+1}: {contradiction}")
                print(f"      Between claims: {pair} | Severity: {severity}")
        else:
            print("\n  [OK] No internal contradictions detected.")

        # ════════════════════════════════════════
        # STEP 5: Final Verdict
        # ════════════════════════════════════════
        print("\n" + "=" * 60)
        print("  Step 5: Final Verdict — Hallucinated / Not Hallucinated")
        print("=" * 60)

        # Build per-finding verdict
        detailed_findings = []
        for i in range(len(findings)):
            cls = classifications[i]
            reason = baseline_reasoning[i]

            # Start with baseline confidence
            if cls.lower() == "typical":
                grounding_score = 1.0
            else:
                # Atypical: check if PubMed grounded it
                if pubmed_evidence.get(i):
                    grounding_score = 0.7  # atypical but has PubMed support
                else:
                    grounding_score = 0.3  # atypical and ungrounded

            # Check if this finding is involved in a contradiction
            is_contradicted = False
            contradiction_details = []
            for ci, pair in enumerate(contradiction_pairs):
                if i in pair:
                    is_contradicted = True
                    severity = severity_scores[ci]
                    grounding_score = max(0.0, grounding_score - severity)
                    contradiction_details.append(
                        contradictions[ci] if ci < len(contradictions) else "Contradiction detected"
                    )
            contradiction_detail = "; ".join(contradiction_details)

            is_hallucination = grounding_score < 0.5

            detailed_findings.append(
                {
                    "finding": findings[i],
                    "target": anatomical_targets[i],
                    "status": clinical_statuses[i],
                    "baseline_classification": cls,
                    "baseline_reasoning": reason,
                    "pubmed_grounded": bool(pubmed_evidence.get(i)),
                    "is_contradicted": is_contradicted,
                    "contradiction_detail": contradiction_detail,
                    "grounding_score": grounding_score,
                    "hallucination_risk": is_hallucination,
                }
            )

            # Update KG confidence
            finding_name = f"Claim_{i+1}"
            if finding_name in self.kg.entities:
                self.kg.entities[finding_name].confidence = grounding_score

            # Update edge confidence
            target_name = anatomical_targets[i].title()
            if target_name in self.kg.graph and finding_name in self.kg.graph[target_name]:
                self.kg.graph[target_name][finding_name]["confidence"] = grounding_score

        # Calculate overall verdict
        if detailed_findings:
            overall_confidence = sum(f["grounding_score"] for f in detailed_findings) / len(detailed_findings)
            hallucination_detected = any(f["hallucination_risk"] for f in detailed_findings)
        else:
            overall_confidence = 0.0
            hallucination_detected = False

        # Print per-finding verdict
        for i, item in enumerate(detailed_findings):
            verdict = "[FAIL] HALLUCINATED" if item["hallucination_risk"] else "[PASS] NOT HALLUCINATED"
            print(f"\n    Claim {i+1}: {verdict}  (Confidence: {item['grounding_score']:.2f})")
            print(f"      \"{item['finding']}\"")
            print(f"      Baseline: {item['baseline_classification']} | PubMed Grounded: {item['pubmed_grounded']} | Contradicted: {item['is_contradicted']}")
            if item["contradiction_detail"]:
                print(f"      Contradiction: {item['contradiction_detail']}")

        # Print overall verdict
        overall_verdict = "[FAIL] HALLUCINATION DETECTED" if hallucination_detected else "[PASS] REPORT IS CLINICALLY CONSISTENT"
        print(f"\n  {'-' * 50}")
        print(f"  OVERALL VERDICT: {overall_verdict}")
        print(f"  Average Confidence Score: {overall_confidence:.2f}")
        print(f"  Graph Stats: {len(self.kg.entities)} entities, {len(self.kg.relations)} relations")
        print(f"  {'-' * 50}")

        return {
            "report": report,
            "overall_confidence": overall_confidence,
            "hallucination_detected": hallucination_detected,
            "detailed_findings": detailed_findings,
            "contradictions": contradictions,
            "graph_stats": {
                "num_entities": len(self.kg.entities),
                "num_relations": len(self.kg.relations),
            },
        }





In [ ]:
system_a = AMG_RAG_ReportSystem(google_api_key=GOOGLE_API_KEY)
print("System A initialized.")


### 1.4 Labeled test set

In [ ]:
test_reports = [
    # --- A. Genuine contradictions (same structure, opposite claims) ---
    {"report": "The heart size is within normal limits. Marked cardiomegaly is present, the heart is grossly enlarged. Lungs are clear bilaterally.", "expected_label": 1},
    {"report": "Frontal and lateral chest radiographs. There is no evidence of pleural effusion. A large left pleural effusion is identified. No pneumothorax is seen.", "expected_label": 1},
    {"report": "Both lung fields are clear without focal opacity. Extensive airspace consolidation is present in the left upper lobe, compatible with pneumonia. Heart size is normal.", "expected_label": 1},
    {"report": "No pneumothorax is identified. A moderate right-sided pneumothorax is seen at the apex. The cardiomediastinal silhouette is unremarkable.", "expected_label": 1},
    {"report": "The mediastinal contour is normal in width. There is marked mediastinal widening, raising concern for aortic injury. The lungs are clear.", "expected_label": 1},
    {"report": "No fracture is seen on this study. An acute, displaced fracture of the left 5th rib is identified. Lungs are clear bilaterally.", "expected_label": 1},
    {"report": "The trachea is midline without deviation. The trachea is deviated markedly to the right. Heart size is within normal limits.", "expected_label": 1},
    {"report": "No acute cardiopulmonary process is identified. There is acute pulmonary edema with diffuse bilateral airspace opacities. The heart is normal in size.", "expected_label": 1},
    {"report": "The osseous structures are intact without fracture. A pathologic compression fracture of the T8 vertebral body is noted. Lungs are clear.", "expected_label": 1},

    # --- B. Self-contradictory single statements ---
    {"report": "The lungs are clear bilaterally with diffuse bilateral ground-glass consolidation. Heart size is normal.", "expected_label": 1},
    {"report": "Heart size is normal in caliber; nonetheless, there is severe, massive cardiac enlargement. Lungs are otherwise unremarkable.", "expected_label": 1},
    {"report": "There is no evidence of pleural fluid; small-to-moderate bilateral pleural effusions are present. Lungs are otherwise clear.", "expected_label": 1},

    # --- C. Physiologically / anatomically impossible ---
    {"report": "The patient is noted to have four lobes within the right lung. Heart size is normal.", "expected_label": 1},
    {"report": "Both kidneys are surgically absent; there is, however, mild enlargement of the left kidney. Lungs are clear.", "expected_label": 1},
    {"report": "The heart demonstrates five distinct cardiac chambers on this study. Lungs are clear bilaterally.", "expected_label": 1},

    # --- D. Near-misses that should NOT be flagged ---
    {"report": "Heart size is normal. Tortuous and mildly prominent thoracic aorta, likely degenerative. Lungs are clear.", "expected_label": 0},
    {"report": "Lungs are clear of consolidation. Mild bibasilar atelectasis is present. Heart size is within normal limits.", "expected_label": 0},
    {"report": "No acute findings. A stable, densely calcified granuloma is again seen, unchanged from prior examination. Heart size is normal.", "expected_label": 0},
    {"report": "No focal consolidation to suggest pneumonia. A small incidental pulmonary nodule is noted; follow-up recommended. Lungs otherwise clear.", "expected_label": 0},
    {"report": "Heart size is normal. Mildly tortuous descending thoracic aorta, likely chronic and age-related. Lungs are clear.", "expected_label": 0},
    {"report": "No acute fracture identified. Old, well-healed fracture deformity of the right clavicle. Lungs are clear bilaterally.", "expected_label": 0},
    {"report": "Lungs are clear without consolidation. Mild chronic interstitial changes, stable in appearance. Heart size is normal.", "expected_label": 0},
    {"report": "No acute cardiopulmonary abnormality. Prominent but stable pulmonary vasculature, unchanged from prior studies. Heart size is normal.", "expected_label": 0},

    # --- E. Ambiguous / severity calibration cases ---
    {"report": "No acute findings. Chronic, longstanding prominence of the pulmonary arteries, stable compared to remote priors. Heart size is normal.", "expected_label": 0},
    {"report": "The heart is top-normal in size. Borderline cardiac enlargement is noted, unchanged from six months ago. Lungs are clear.", "expected_label": 0},
    {"report": "No acute findings. There is new, interval enlargement of the pulmonary trunk compared to the prior study performed one week ago. Lungs are clear.", "expected_label": 1},
]

print(f"Total test reports: {len(test_reports)}")
print(f"Positive (should flag): {sum(r['expected_label'] for r in test_reports)}")
print(f"Negative (should not flag): {sum(1 - r['expected_label'] for r in test_reports)}")


### 1.5 Run System A over the test set, timing each report

In [ ]:
import time
import io
import contextlib
from tqdm import tqdm

system_a_results = []

for i, item in enumerate(tqdm(test_reports, desc="System A")):
    report_text = item["report"]
    expected = item["expected_label"]

    start = time.perf_counter()
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            result = system_a.evaluate_medical_report(report_text)
        elapsed = time.perf_counter() - start
        error = None
        predicted = int(result["hallucination_detected"])
        confidence = result.get("overall_confidence", None)
    except Exception as e:
        elapsed = time.perf_counter() - start
        error = str(e)
        predicted = None
        confidence = None

    system_a_results.append({
        "report_id": i + 1,
        "report": report_text,
        "expected_label": expected,
        "predicted_label": predicted,
        "overall_confidence": confidence,
        "latency_sec": elapsed,
        "error": error,
    })

print(f"Done. {sum(1 for r in system_a_results if r['error'])} report(s) errored out.")


### 1.6 Save System A's results to disk

In [ ]:
import json

with open("system_a_results.json", "w") as f:
    json.dump(system_a_results, f, indent=2)

print("Saved system_a_results.json")
print("\nNow: Runtime -> Restart runtime, then continue with Part 2 below.")


---
# Part 2: System B (Single-Verifier Pipeline)

Start this part fresh, after restarting the runtime following Part 1.
This part does not depend on any variables from Part 1 -- it's fully self-contained.

### 2.1 Install dependencies

In [ ]:
%pip install -q "google-auth==2.49.0" "langchain-google-genai" "langchain-classic" "langchain-huggingface" "langchain-chroma" networkx wikipedia python-decouple
%pip install -q pandas tqdm scikit-learn


### 2.2 API key

System B now uses the same `GOOGLE_API_KEY` and constructor style as System A.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")

pubmed_key = getpass("Enter your PubMed API key (optional, press Enter to skip): ")
if pubmed_key:
    os.environ["pubmed_api"] = pubmed_key

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
print("API key configured.")


### 2.3 System B source

In [ ]:
"""
AMG-RAG: Autonomous Medical Knowledge Graph RAG System
Complete implementation with dynamic KG generation and medical QA
"""

import json
import os
import time
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import networkx as nx
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import requests
from xml.etree import ElementTree as ET
import wikipedia
from typing_extensions import TypedDict
from decouple import config
# Configuration - Replace with your API keys
GOOGLE_API_KEY = config('GOOGLE_API_KEY', default=os.environ.get('GOOGLE_API_KEY'))
PUBMED_API_KEY = config('pubmed_api', default=os.environ.get('pubmed_api') or None)

@dataclass
class MedicalEntity:
    """Represents a medical entity in the knowledge graph"""
    name: str
    description: str
    entity_type: str  # drug, disease, symptom, treatment, etc.
    confidence: float = 1.0
    sources: List[str] = field(default_factory=list)

@dataclass
class MedicalRelation:
    """Represents a relationship between medical entities"""
    source: str
    target: str
    relation_type: str
    confidence: float
    evidence: str
    sources: List[str] = field(default_factory=list)

class MedicalKnowledgeGraph:
    """Dynamic Medical Knowledge Graph with confidence scoring"""

    def __init__(self):
        self.graph = nx.DiGraph()
        self.entities = {}
        self.relations = []

    def add_entity(self, entity: MedicalEntity):
        """Add a medical entity to the graph"""
        self.entities[entity.name] = entity
        self.graph.add_node(
            entity.name,
            description=entity.description,
            entity_type=entity.entity_type,
            confidence=entity.confidence,
            sources=entity.sources
        )

    def add_relation(self, relation: MedicalRelation):
        """Add a relationship between entities"""
        self.relations.append(relation)
        self.graph.add_edge(
            relation.source,
            relation.target,
            relation_type=relation.relation_type,
            confidence=relation.confidence,
            evidence=relation.evidence,
            sources=relation.sources
        )

    def get_connected_nodes(self, node_name: str, confidence_threshold: float = 0.5):
        """Get nodes connected to a given node with confidence above threshold"""
        connected = []
        if node_name in self.graph:
            for neighbor in self.graph.neighbors(node_name):
                edge_data = self.graph[node_name][neighbor]
                if edge_data.get('confidence', 0) >= confidence_threshold:
                    connected.append({
                        'node': neighbor,
                        'relation': edge_data.get('relation_type'),
                        'confidence': edge_data.get('confidence'),
                        'evidence': edge_data.get('evidence')
                    })
        return connected

    def explore_path(self, start_node: str, max_depth: int = 3,
                    confidence_threshold: float = 0.5):
        """Explore paths from a starting node with confidence propagation"""
        paths = []
        visited = set()

        def dfs(node, path, accumulated_confidence, depth):
            if depth > max_depth or node in visited:
                return

            visited.add(node)

            if len(path) > 0:
                paths.append({
                    'path': path.copy(),
                    'confidence': accumulated_confidence,
                    'final_node': node
                })

            for neighbor_data in self.get_connected_nodes(node, confidence_threshold):
                neighbor = neighbor_data['node']
                new_confidence = accumulated_confidence * neighbor_data['confidence']

                if new_confidence >= confidence_threshold:
                    new_path = path + [(node, neighbor, neighbor_data['relation'])]
                    dfs(neighbor, new_path, new_confidence, depth + 1)

            visited.remove(node)

        dfs(start_node, [], 1.0, 0)
        return paths

class PubMedSearcher:
    """PubMed API wrapper for medical literature search"""

    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key
        self.base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    def search(self, query: str, max_results: int = 3) -> List[str]:
        """Search PubMed and return article abstracts"""
        # Respect rate limits for free tier (3 requests/sec max)
        time.sleep(0.5)

        # Search for PMIDs
        search_url = f"{self.base_url}/esearch.fcgi"
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmode": "xml",
            "retmax": max_results
        }
        if self.api_key:
            search_params["api_key"] = self.api_key

        try:
            response = requests.get(search_url, params=search_params, timeout=30)
            if response.status_code != 200:
                print(f"PubMed API search returned HTTP status {response.status_code}")
                return []

            if not response.text.strip().startswith("<?xml") and not response.text.strip().startswith("<"):
                print(f"PubMed API returned non-XML response during search: {response.text[:200]}")
                return []

            root = ET.fromstring(response.text)
            pmids = [id_elem.text for id_elem in root.findall(".//Id")]

            if not pmids:
                return []

            # Fetch abstracts
            time.sleep(0.5)
            fetch_url = f"{self.base_url}/efetch.fcgi"
            fetch_params = {
                "db": "pubmed",
                "id": ",".join(pmids),
                "retmode": "text",
                "rettype": "abstract"
            }
            if self.api_key:
                fetch_params["api_key"] = self.api_key

            response = requests.get(fetch_url, params=fetch_params, timeout=30)
            if response.status_code != 200:
                print(f"PubMed API fetch returned HTTP status {response.status_code}")
                return []

            articles = response.text.split("\n\n")

            # Clean and return abstracts
            abstracts = []
            for article in articles:
                lines = article.split("\n")
                abstract_lines = [line for line in lines if line.strip()
                                and not any(skip in line.lower() for skip in
                                          ["author", "doi", "pmid", "copyright"])]
                if abstract_lines:
                    abstracts.append(" ".join(abstract_lines))

            return abstracts

        except Exception as e:
            print(f"PubMed search error: {e}")
            return []

class AMG_RAG_System:
    """Main AMG-RAG system for medical question answering"""

    def __init__(self, google_api_key: str = None):
        # Initialize LLM with the same Gemini/Gemma backend and model as
        # System A, so both systems are evaluated on identical LLM output.
        if google_api_key:
            self.llm = ChatGoogleGenerativeAI(
                model="gemma-4-31b-it",
                temperature=0.0,
                google_api_key=google_api_key,
            )
        else:
            raise ValueError("Google API key is required.")

        # Initialize components
        self.kg = MedicalKnowledgeGraph()
        self.pubmed = PubMedSearcher(api_key=PUBMED_API_KEY)
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vector_store = Chroma(
            collection_name="medical_qa",
            embedding_function=self.embeddings
        )

        # Initialize chains
        self._setup_chains()

    def _setup_chains(self):
        """Setup LLM chains for various tasks"""

        # Enhanced medical entity extraction with relevance scoring
        entity_schemas = [
            ResponseSchema(
                name="entities",
                description="List of medical entities (diseases, drugs, symptoms, treatments)",
                type="array"
            ),
            ResponseSchema(
                name="scores",
                description="Relevance scores (1-10) for each entity based on importance to the question",
                type="array"
            ),
            ResponseSchema(
                name="descriptions",
                description="Brief descriptions of each entity in the context of the question",
                type="array"
            )
        ]
        entity_parser = StructuredOutputParser.from_response_schemas(entity_schemas)

        self.entity_extractor = PromptTemplate(
            template="""Extract all medical entities from this question and options with relevance scoring.
            Include diseases, drugs, symptoms, treatments, and medical concepts.

            Question: {question}
            Options: {options}
            Context: {context}

            For each entity, provide:
            1. Entity name
            2. Relevance score (1-10): 10=directly related to question, 7-9=moderately relevant, 4-6=weakly relevant, 1-3=minimally relevant
            3. Brief description in context of the question

            Return in JSON format:
            {format_instructions}""",
            input_variables=["question", "options", "context"],
            partial_variables={"format_instructions": entity_parser.get_format_instructions()}
        ) | self.llm | entity_parser

        # Enhanced relation extraction with bidirectional analysis
        relation_schemas = [
            ResponseSchema(
                name="relationships",
                description="List of relationship dictionaries between entities containing entityA, entityB, relationship_type, confidence_score, and evidence",
                type="array"
            )
        ]
        relation_parser = StructuredOutputParser.from_response_schemas(relation_schemas)

        self.relation_extractor = PromptTemplate(
            template="""Analyze the medical relationships between these entities based on the context.

            Entities and Descriptions:
            {entities_with_descriptions}

            Context: {context}

            Identify and return all valid medical relationships between any of the given entities based on the medical context.

            Provide relationships in this exact JSON format:
            {{
                "relationships": [
                    {{
                        "entityA": "Entity Name 1",
                        "entityB": "Entity Name 2",
                        "relationship_type": "relationship_type_here",
                        "confidence_score": 8,
                        "evidence": "brief evidence here"
                    }}
                ]
            }}

            Use medical relationship types like: treats, causes, symptom_of, risk_factor_for, contraindicated_with, differential_diagnosis, etc.
            Confidence scores: 10=strong evidence, 7-9=moderate evidence, 4-6=weak evidence, 1-3=minimal evidence

            Return ONLY the JSON, no other text:""",
            input_variables=["entities_with_descriptions", "context"],
            partial_variables={"format_instructions": relation_parser.get_format_instructions()}
        ) | self.llm | relation_parser

        # Entity summarization chain
        summary_schemas = [
            ResponseSchema(
                name="summaries",
                description="Concise summaries for each entity based on context",
                type="array"
            ),
            ResponseSchema(
                name="scores",
                description="Relevance scores (1-10) for each summary",
                type="array"
            )
        ]
        summary_parser = StructuredOutputParser.from_response_schemas(summary_schemas)

        self.summary_chain = PromptTemplate(
            template="""Generate concise and relevant summaries for each medical entity based on the given context.

            Entities: {entities}
            Context: {context}

            For each entity, provide:
            1. A concise summary (2-3 sentences) focusing on relevance to the medical question
            2. Relevance score (1-10): 10=directly relevant, 7-9=moderately relevant, 4-6=weakly relevant, 1-3=minimally relevant

            Return in JSON format:
            {format_instructions}""",
            input_variables=["entities", "context"],
            partial_variables={"format_instructions": summary_parser.get_format_instructions()}
        ) | self.llm | summary_parser

        # Chain of thought reasoning
        cot_schemas = [
            ResponseSchema(
                name="reasoning",
                description="Step-by-step medical reasoning",
                type="string"
            )
        ]
        cot_parser = StructuredOutputParser.from_response_schemas(cot_schemas)

        self.cot_chain = PromptTemplate(
            template="""Based on the medical knowledge graph information and search results,
            provide step-by-step reasoning for this medical question.

            Question: {question}

            Graph Knowledge:
            {graph_context}

            Search Results:
            {search_context}

            Provide detailed medical reasoning:
            {format_instructions}""",
            input_variables=["question", "graph_context", "search_context"],
            partial_variables={"format_instructions": cot_parser.get_format_instructions()}
        ) | self.llm | cot_parser

        # Final answer generation
        answer_schemas = [
            ResponseSchema(
                name="answer",
                description="Final answer (A, B, C, D, or E)",
                type="string"
            ),
            ResponseSchema(
                name="confidence",
                description="Confidence in the answer (0-1)",
                type="number"
            ),
            ResponseSchema(
                name="explanation",
                description="Brief explanation",
                type="string"
            )
        ]
        answer_parser = StructuredOutputParser.from_response_schemas(answer_schemas)

        self.answer_chain = PromptTemplate(
            template="""Based on the reasoning and evidence, select the best answer.

            Question: {question}
            Options: {options}

            Reasoning:
            {reasoning}

            Evidence:
            {evidence}

            Select the best answer (A, B, C, D, or E):
            {format_instructions}""",
            input_variables=["question", "options", "reasoning", "evidence"],
            partial_variables={"format_instructions": answer_parser.get_format_instructions()}
        ) | self.llm | answer_parser

        # Report findings extractor
        finding_schemas = [
            ResponseSchema(
                name="findings",
                description="List of clinical claims/findings parsed from the report (e.g. 'heart is of normal size', 'lungs are clear')",
                type="array"
            ),
            ResponseSchema(
                name="anatomical_targets",
                description="The anatomical structures associated with each finding (e.g. 'heart', 'lungs', 'mediastinum')",
                type="array"
            ),
            ResponseSchema(
                name="clinical_status",
                description="Clinical observation status for each finding (e.g. 'normal', 'abnormal', 'clear')",
                type="array"
            )
        ]
        finding_parser = StructuredOutputParser.from_response_schemas(finding_schemas)

        self.finding_extractor = PromptTemplate(
            template="""Extract all individual medical findings/claims and associated anatomical structures from this clinical report.

            Report: {report}

            For each finding/claim, provide:
            1. The finding statement
            2. The anatomical target structure
            3. The clinical status (e.g. normal, abnormal, clear, consolidated, etc.)

            Return in JSON format:
            {format_instructions}""",
            input_variables=["report"],
            partial_variables={"format_instructions": finding_parser.get_format_instructions()}
        ) | self.llm | finding_parser

        # Report verifier schema
        verifier_schemas = [
            ResponseSchema(
                name="grounding_scores",
                description="Grounding/confidence score (0.0 to 1.0) for each finding based on clinical consistency and supporting context. 1.0 = highly plausible/normal medical finding, 0.5 = questionable/vague, 0.0 = highly contradictory or medically impossible.",
                type="array"
            ),
            ResponseSchema(
                name="assessments",
                description="Brief clinical validation reasoning for each finding's score, checking if the claim aligns with typical medical findings and standard terminology.",
                type="array"
            ),
            ResponseSchema(
                name="hallucination_indicators",
                description="Boolean flag for each finding: true if the finding appears contradictory, clinically impossible, or likely an LLM hallucination; false otherwise.",
                type="array"
            )
        ]
        verifier_parser = StructuredOutputParser.from_response_schemas(verifier_schemas)

        self.finding_verifier = PromptTemplate(
            template="""Validate the following clinical findings/claims against the provided literature context and general medical knowledge.
            Determine if they are medically consistent, physiologically plausible, and well-grounded.

            Findings: {findings_list}
            Literature & Evidence Context: {context}

            For each finding, provide:
            1. Grounding score (0.0 to 1.0): 1.0 = standard, highly consistent finding; 0.7-0.9 = plausible; 0.4-0.6 = weak grounding or inconsistent; 0.0-0.3 = clinically contradictory or medical hallucination.
            2. Brief assessment reasoning.
            3. Hallucination indicator (true/false).

            Return in JSON format:
            {format_instructions}""",
            input_variables=["findings_list", "context"],
            partial_variables={"format_instructions": verifier_parser.get_format_instructions()}
        ) | self.llm | verifier_parser

    def build_knowledge_graph(self, question: str, options: Dict[str, str]) -> None:
        """Build a dynamic knowledge graph for the question with enhanced entity extraction"""

        # Prepare context for entity extraction
        options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
        full_text = question + " " + " ".join(options.values())

        # Search for additional context
        search_query = question + " " + " ".join(list(options.values())[:3])
        search_results = self.pubmed.search(search_query, max_results=3)
        context = "\n".join(search_results) if search_results else ""

        # Extract medical entities with relevance scoring
        try:
            entities_result = self.entity_extractor.invoke({
                "question": question,
                "options": options_text,
                "context": context
            })
            entities = entities_result.get("entities", [])
            scores = entities_result.get("scores", [])
            descriptions = entities_result.get("descriptions", [])
        except Exception as e:
            print(f"Entity extraction error: {e}")
            entities = list(options.values())[:3]  # Fallback to options
            scores = [5] * len(entities)  # Default moderate relevance
            descriptions = [f"Medical concept: {entity}" for entity in entities]

        print(f"Extracted entities: {entities}")
        print(f"Relevance scores: {scores}")

        # Add entities to graph with relevance-based confidence
        for i, entity in enumerate(entities[:8]):  # Limit to 8 entities
            # Search PubMed for additional information
            abstracts = self.pubmed.search(entity, max_results=2)

            # Search Wikipedia as fallback
            wiki_content = ""
            try:
                wiki_results = wikipedia.search(entity, results=1)
                if wiki_results:
                    wiki_content = wikipedia.summary(wiki_results[0], sentences=3)
            except:
                pass

            # Combine sources with LLM-generated description
            llm_description = descriptions[i] if i < len(descriptions) else f"Medical entity: {entity}"
            external_description = " ".join(abstracts) if abstracts else wiki_content
            combined_description = f"{llm_description}. {external_description}" if external_description else llm_description

            # Calculate confidence based on relevance score and external sources
            relevance_score = scores[i] if i < len(scores) else 5
            confidence = min(1.0, (relevance_score / 10.0) + (0.2 if abstracts else 0.1))

            # Add entity to graph
            med_entity = MedicalEntity(
                name=entity,
                description=combined_description[:500],  # Limit description length
                entity_type="medical_concept",
                confidence=confidence,
                sources=["PubMed", "Wikipedia"] if abstracts else ["Wikipedia"]
            )
            self.kg.add_entity(med_entity)

        # Extract relationships between entities in one batch for maximum speed
        entity_list = list(self.kg.entities.keys())
        if len(entity_list) > 1:
            try:
                # Prepare descriptions block
                desc_list = []
                for entity in entity_list:
                    desc_list.append(f"- {entity}: {self.kg.entities[entity].description[:200]}")
                entities_with_descriptions = "\n".join(desc_list)

                relationship_context = f"Question: {question}\n\nOptions: {options_text}\n\nSearch Results: {context}"

                print("Extracting relationships in a single batch...")
                relation_result = self.relation_extractor.invoke({
                    "entities_with_descriptions": entities_with_descriptions,
                    "context": relationship_context
                })

                relationships = relation_result.get("relationships", [])

                # Process each relationship in the list
                for rel in relationships:
                    if isinstance(rel, dict):
                        rel_type = rel.get("relationship_type", "related_to")
                        confidence = rel.get("confidence_score", 5) / 10.0
                        evidence = rel.get("evidence", "")
                        entity_a = rel.get("entityA", "")
                        entity_b = rel.get("entityB", "")

                        # Validate that both entities exist in our graph to avoid hallucinations
                        if entity_a in self.kg.entities and entity_b in self.kg.entities:
                            relation = MedicalRelation(
                                source=entity_a,
                                target=entity_b,
                                relation_type=rel_type,
                                confidence=confidence,
                                evidence=evidence,
                                sources=["LLM Analysis"]
                            )
                            self.kg.add_relation(relation)

            except Exception as e:
                print(f"Relation extraction error: {e}")

        # Generate entity summaries for better context
        self._generate_entity_summaries(question, context)

    def _generate_entity_summaries(self, question: str, context: str) -> None:
        """Generate enhanced summaries for entities in the knowledge graph"""
        if not self.kg.entities:
            return

        try:
            entities_list = list(self.kg.entities.keys())
            summary_result = self.summary_chain.invoke({
                "entities": entities_list,
                "context": f"Question: {question}\n\nContext: {context}"
            })

            summaries = summary_result.get("summaries", [])
            scores = summary_result.get("scores", [])

            # Update entity descriptions with enhanced summaries
            for i, entity_name in enumerate(entities_list):
                if i < len(summaries) and i < len(scores):
                    # Combine original description with enhanced summary
                    original_desc = self.kg.entities[entity_name].description
                    enhanced_summary = summaries[i]
                    relevance_score = scores[i]

                    # Update description with enhanced summary
                    updated_description = f"{original_desc}\n\nEnhanced Summary: {enhanced_summary}"

                    # Update confidence based on summary relevance
                    current_confidence = self.kg.entities[entity_name].confidence
                    summary_confidence = min(1.0, relevance_score / 10.0)
                    updated_confidence = min(1.0, (current_confidence + summary_confidence) / 2)

                    # Update the entity
                    self.kg.entities[entity_name].description = updated_description[:500]
                    self.kg.entities[entity_name].confidence = updated_confidence

        except Exception as e:
            print(f"Entity summarization error: {e}")

    def reason_with_graph(self, question: str, options: Dict[str, str]) -> Dict[str, Any]:
        """Perform reasoning using the knowledge graph"""

        # Explore graph paths for each entity
        graph_context = []
        for entity in list(self.kg.entities.keys())[:3]:
            # Get connected nodes
            connections = self.kg.get_connected_nodes(entity, confidence_threshold=0.3)

            # Explore paths
            paths = self.kg.explore_path(entity, max_depth=2, confidence_threshold=0.3)

            context = f"Entity: {entity}\n"
            context += f"Description: {self.kg.entities[entity].description[:200]}\n"

            if connections:
                context += "Direct connections:\n"
                for conn in connections[:3]:
                    context += f"  - {conn['relation']} -> {conn['node']} (confidence: {conn['confidence']:.2f})\n"

            if paths:
                context += "Reasoning paths:\n"
                for path_data in paths[:2]:
                    path_str = " -> ".join([f"{p[0]} [{p[2]}]" for p in path_data['path']])
                    if path_str:
                        context += f"  - {path_str} -> {path_data['final_node']} (confidence: {path_data['confidence']:.2f})\n"

            graph_context.append(context)

        # Search for additional evidence
        search_query = question + " " + " ".join(list(self.kg.entities.keys())[:3])
        search_results = self.pubmed.search(search_query, max_results=2)
        search_context = "\n".join(search_results) if search_results else "No additional search results found."

        # Generate chain of thought reasoning
        try:
            cot_result = self.cot_chain.invoke({
                "question": question,
                "graph_context": "\n\n".join(graph_context),
                "search_context": search_context
            })
            reasoning = cot_result.get("reasoning", "Unable to generate reasoning")
        except Exception as e:
            print(f"CoT generation error: {e}")
            reasoning = "Error in reasoning generation"

        # Generate final answer
        options_str = "\n".join([f"{k}: {v}" for k, v in options.items()])
        evidence = "\n".join(graph_context[:2])

        try:
            answer_result = self.answer_chain.invoke({
                "question": question,
                "options": options_str,
                "reasoning": reasoning,
                "evidence": evidence
            })

            return {
                "answer": answer_result.get("answer", "Unable to determine"),
                "confidence": answer_result.get("confidence", 0.0),
                "explanation": answer_result.get("explanation", ""),
                "reasoning": reasoning,
                "graph_context": graph_context,
                "search_context": search_context
            }
        except Exception as e:
            print(f"Answer generation error: {e}")
            return {
                "answer": "Error",
                "confidence": 0.0,
                "explanation": str(e),
                "reasoning": reasoning,
                "graph_context": graph_context,
                "search_context": search_context
            }

    def evaluate_medical_report(self, report: str) -> Dict[str, Any]:
        """Evaluate an LLM-generated medical report for clinical consistency, hallucinations, and confidence scoring."""
        print(f"\n{'='*50}")
        print("Step 1: Parsing report into clinical claims...")

        # 1. Extract findings
        try:
            finding_result = self.finding_extractor.invoke({"report": report})
            findings = finding_result.get("findings", [])
            anatomical_targets = finding_result.get("anatomical_targets", [])
            clinical_statuses = finding_result.get("clinical_status", [])
        except Exception as e:
            print(f"Finding extraction error: {e}")
            # Fallback parsing
            findings = [s.strip() for s in report.split(".") if s.strip()]
            anatomical_targets = ["General"] * len(findings)
            clinical_statuses = ["Unspecified"] * len(findings)

        # Open-weight/prompt-based JSON extraction doesn't guarantee the three
        # parallel arrays come back the same length (unlike a strict schema-
        # enforced provider). Truncate to the shortest list and pad any gaps
        # with safe defaults so every downstream index is always in range.
        n = min(len(findings), len(anatomical_targets), len(clinical_statuses)) if findings and anatomical_targets and clinical_statuses else 0
        if n < len(findings):
            print(f"Warning: extractor returned mismatched array lengths "
                  f"(findings={len(findings)}, targets={len(anatomical_targets)}, "
                  f"statuses={len(clinical_statuses)}) — truncating to {n} aligned claims.")
        findings = findings[:n]
        anatomical_targets = (anatomical_targets[:n] + ["General"] * n)[:n]
        clinical_statuses = (clinical_statuses[:n] + ["Unspecified"] * n)[:n]

        print(f"Extracted {len(findings)} clinical claims.")

        # 2. Build Report Knowledge Graph
        # Clear existing graph first to avoid contamination
        self.kg = MedicalKnowledgeGraph()

        for i, target in enumerate(anatomical_targets):
            if i < len(findings):
                # Add anatomical target as an entity with high confidence
                entity_name = target.title()
                if entity_name not in self.kg.entities:
                    med_entity = MedicalEntity(
                        name=entity_name,
                        description=f"Anatomical region: {entity_name}. Status in report: {clinical_statuses[i]}",
                        entity_type="anatomical_structure",
                        confidence=1.0,
                        sources=["Report Parser"]
                    )
                    self.kg.add_entity(med_entity)

                # Add the specific finding/claim
                finding_name = f"Claim_{i+1}"
                finding_entity = MedicalEntity(
                    name=finding_name,
                    description=findings[i],
                    entity_type="clinical_finding",
                    confidence=1.0,
                    sources=["Report Parser"]
                )
                self.kg.add_entity(finding_entity)

                # Link target to finding
                relation = MedicalRelation(
                    source=entity_name,
                    target=finding_name,
                    relation_type="has_finding",
                    confidence=1.0,
                    evidence=f"Status: {clinical_statuses[i]}",
                    sources=["Report Parser"]
                )
                self.kg.add_relation(relation)

        # 3. Retrieve Context & Validate each claim
        print("Step 2: Performing literature verification on claims...")

        # We search PubMed or Wiki for anatomical targets to get typical reference context
        search_queries = list(set(anatomical_targets))[:3]
        combined_search_query = " ".join(search_queries) + " normal chest x-ray"
        pubmed_articles = self.pubmed.search(combined_search_query, max_results=3)
        context = "\n".join(pubmed_articles) if pubmed_articles else "Standard anatomical reference context for normal chest findings."

        findings_formatted = "\n".join([f"- {findings[i]} (Target: {anatomical_targets[i]}, Status: {clinical_statuses[i]})" for i in range(len(findings))])

        def _safe_float(v, default=0.9):
            try:
                return float(v)
            except (TypeError, ValueError):
                return default

        def _safe_bool(v, default=False):
            if isinstance(v, bool):
                return v
            if isinstance(v, str):
                return v.strip().lower() in ("true", "yes", "1")
            return default

        try:
            verifier_result = self.finding_verifier.invoke({
                "findings_list": findings_formatted,
                "context": context
            })
            grounding_scores = [_safe_float(s) for s in verifier_result.get("grounding_scores", [])]
            assessments = [str(a) for a in verifier_result.get("assessments", [])]
            hallucination_indicators = [_safe_bool(h) for h in verifier_result.get("hallucination_indicators", [])]
        except Exception as e:
            print(f"Finding verification error: {e}")
            grounding_scores = [0.9] * len(findings)
            assessments = ["Standard verification fallback."] * len(findings)
            hallucination_indicators = [False] * len(findings)

        # 4. Compile detailed results and update KG confidence
        detailed_findings = []
        for i in range(len(findings)):
            g_score = grounding_scores[i] if i < len(grounding_scores) else 0.9
            assessment = assessments[i] if i < len(assessments) else "No detailed assessment."
            is_hallucination = hallucination_indicators[i] if i < len(hallucination_indicators) else False

            detailed_findings.append({
                "finding": findings[i],
                "target": anatomical_targets[i],
                "status": clinical_statuses[i],
                "grounding_score": g_score,
                "assessment": assessment,
                "hallucination_risk": is_hallucination
            })

            # Update finding entity confidence in KG
            finding_name = f"Claim_{i+1}"
            if finding_name in self.kg.entities:
                self.kg.entities[finding_name].confidence = g_score
                self.kg.entities[finding_name].description += f" (Verified Score: {g_score:.2f} - {assessment})"

            # Update connection edge confidence in KG
            target_name = anatomical_targets[i].title()
            if target_name in self.kg.graph and finding_name in self.kg.graph[target_name]:
                self.kg.graph[target_name][finding_name]['confidence'] = g_score

        # Calculate overall report confidence
        if detailed_findings:
            overall_confidence = sum([f["grounding_score"] for f in detailed_findings]) / len(detailed_findings)
            hallucination_detected = any([f["hallucination_risk"] for f in detailed_findings])
        else:
            overall_confidence = 0.0
            hallucination_detected = False

        return {
            "report": report,
            "overall_confidence": overall_confidence,
            "hallucination_detected": hallucination_detected,
            "detailed_findings": detailed_findings,
            "graph_stats": {
                "num_entities": len(self.kg.entities),
                "num_relations": len(self.kg.relations)
            }
        }

    def answer_question(self, question_data: Dict[str, Any]) -> Dict[str, Any]:
        """Main pipeline to answer a medical question"""

        question = question_data["question"]
        options = question_data.get("options", {})

        print(f"\n{'='*50}")
        print(f"Question: {question}")
        print(f"Options: {options}")
        print(f"{'='*50}\n")

        # Step 1: Build knowledge graph
        print("Step 1: Building knowledge graph...")
        self.build_knowledge_graph(question, options)
        print(f"Graph has {len(self.kg.entities)} entities and {len(self.kg.relations)} relations")

        # Step 2: Reason with graph
        print("\nStep 2: Reasoning with graph...")
        result = self.reason_with_graph(question, options)

        # Add metadata
        result["question"] = question
        result["options"] = options
        result["expected_answer"] = question_data.get("answer", "Unknown")
        result["graph_stats"] = {
            "num_entities": len(self.kg.entities),
            "num_relations": len(self.kg.relations)
        }

        return result

def load_medqa_sample():
    """Load a sample from MEDQA dataset"""
    # Sample MEDQA question
    sample = {
        "question": "A 45-year-old man presents to the emergency department with severe chest pain that started 2 hours ago. The pain is substernal, crushing in nature, and radiates to his left arm. He has a history of hypertension and diabetes mellitus. His father died of a myocardial infarction at age 50. On examination, he is diaphoretic and in distress. His blood pressure is 150/90 mmHg, pulse is 110/min, and respirations are 22/min. An ECG shows ST-segment elevation in leads II, III, and aVF. Which of the following is the most likely diagnosis?",
        "options": {
            "A": "Unstable angina",
            "B": "Acute inferior wall myocardial infarction",
            "C": "Acute anterior wall myocardial infarction",
            "D": "Aortic dissection",
            "E": "Pulmonary embolism"
        },
        "answer": "B",
        "answer_idx": 1,
        "meta_info": "This is a cardiology question testing knowledge of myocardial infarction presentation and ECG findings."
    }
    return sample

def main():
    """Main execution function"""

    print("AMG-RAG Medical QA System")
    print("="*50)

    # Initialize system
    print("Initializing AMG-RAG system...")

    # Using Google Gemini
    system = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)

    # Load sample question
    print("\nLoading MEDQA sample question...")
    question_data = load_medqa_sample()



    result = system.answer_question(question_data)



    # Display results
    print("\n" + "="*50)
    print("RESULTS")
    print("="*50)
    print(f"Question: {result['question'][:100]}...")
    print(f"\nOptions:")
    for k, v in result['options'].items():
        print(f"  {k}: {v}")

    print(f"\nExpected Answer: {result['expected_answer']}")
    print(f"Model Answer: {result['answer']}")
    print(f"Confidence: {result['confidence']:.2f}")
    print(f"\nExplanation: {result['explanation']}")

    print(f"\nGraph Statistics:")
    print(f"  - Entities: {result['graph_stats']['num_entities']}")
    print(f"  - Relations: {result['graph_stats']['num_relations']}")

    print(f"\nReasoning Chain:")
    print(result['reasoning'][:500] + "..." if len(result['reasoning']) > 500 else result['reasoning'])



    # Visualize graph structure (text-based)
    print("\n" + "="*50)
    print("KNOWLEDGE GRAPH STRUCTURE")
    print("="*50)

    for entity_name, entity in list(system.kg.entities.items())[:5]:
        print(f"\n[*] {entity_name}")
        print(f"   Type: {entity.entity_type}")
        print(f"   Description: {entity.description[:100]}...")

        connections = system.kg.get_connected_nodes(entity_name)
        if connections:
            print("   Connections:")
            for conn in connections[:3]:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

    # New section: Report verification demonstration
    print("\n" + "="*50)
    print("DEMONSTRATION: CLINICAL REPORT VERIFICATION & HALLUCINATION ASSESSMENT")
    print("="*50)

    sample_report = (
        "The heart is of normal size with no abnormalities. The lungs are clear with no signs of "
        "consolidation or fluid accumulation. The structures around the lungs, including the "
        "mediastinum and hilar regions are normal. Overall, there are no acute or concerning findings."
    )

    print(f"Input Chest X-ray Report:\n\"{sample_report}\"")

    report_result = system.evaluate_medical_report(sample_report)

    print("\n" + "="*50)
    print("REPORT VERIFICATION RESULTS")
    print("="*50)
    print(f"Overall Report Confidence: {report_result['overall_confidence']:.2f}")
    print(f"Hallucination / Critical Risk Detected: {report_result['hallucination_detected']}")

    print("\nDetailed Findings Breakdown:")
    for idx, item in enumerate(report_result['detailed_findings']):
        print(f"\nFinding #{idx+1}:")
        print(f"  - Statement: \"{item['finding']}\"")
        print(f"  - Anatomical Target: {item['target']}")
        print(f"  - Clinical Status: {item['status']}")
        print(f"  - Grounding/Confidence Score: {item['grounding_score']:.2f}")
        print(f"  - Verification Assessment: {item['assessment']}")
        print(f"  - Hallucination Risk: {item['hallucination_risk']}")

    print("\n" + "="*50)
    print("FINDINGS GRAPH STRUCTURE")
    print("="*50)
    for entity_name, entity in list(system.kg.entities.items()):
        print(f"\n[*] {entity_name} ({entity.entity_type})")
        print(f"   Confidence: {entity.confidence:.2f}")
        print(f"   Details: {entity.description[:150]}")
        connections = system.kg.get_connected_nodes(entity_name, confidence_threshold=0.0)
        if connections:
            print("   Connections:")
            for conn in connections:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

    # Demonstration 2: Hallucinated/Contradictory Report
    print("\n" + "="*50)
    print("DEMONSTRATION 2: HALLUCINATED / CONTRADICTORY REPORT")
    print("="*50)

    hallucinated_report = (
        "The heart is of normal size with signs of severe acute cardiomegaly. The lungs show massive "
        "consolidations in the upper lobes with completely clear lung fields. The mediastinum is "
        "widened and perfectly normal."
    )

    print(f"Input Hallucinated Report:\n\"{hallucinated_report}\"")

    hallucinated_result = system.evaluate_medical_report(hallucinated_report)

    print("\n" + "="*50)
    print("REPORT VERIFICATION RESULTS (HALLUCINATED REPORT)")
    print("="*50)
    print(f"Overall Report Confidence: {hallucinated_result['overall_confidence']:.2f}")
    print(f"Hallucination / Critical Risk Detected: {hallucinated_result['hallucination_detected']}")

    print("\nDetailed Findings Breakdown:")
    for idx, item in enumerate(hallucinated_result['detailed_findings']):
        print(f"\nFinding #{idx+1}:")
        print(f"  - Statement: \"{item['finding']}\"")
        print(f"  - Anatomical Target: {item['target']}")
        print(f"  - Clinical Status: {item['status']}")
        print(f"  - Grounding/Confidence Score: {item['grounding_score']:.2f}")
        print(f"  - Verification Assessment: {item['assessment']}")
        print(f"  - Hallucination Risk: {item['hallucination_risk']}")

    print("\n" + "="*50)
    print("FINDINGS GRAPH STRUCTURE (HALLUCINATED)")
    print("="*50)
    for entity_name, entity in list(system.kg.entities.items()):
        print(f"\n[*] {entity_name} ({entity.entity_type})")
        print(f"   Confidence: {entity.confidence:.2f}")
        print(f"   Details: {entity.description[:150]}")
        connections = system.kg.get_connected_nodes(entity_name, confidence_threshold=0.0)
        if connections:
            print("   Connections:")
            for conn in connections:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

# NOTE: main() is a self-contained CLI-style demo (MEDQA sample question +
# 2 hardcoded report demos). It's left here for reference but is NOT auto-run,
# so importing this cell doesn't trigger extra LLM calls / cost.
# Uncomment the next two lines if you want to run that demo:
# if __name__ == "__main__":
#     main()

# Initialize the system for use in the rest of this notebook.
# Uses the GOOGLE_API_KEY set in the cell above (edit that cell, not this one).
rag_system = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)
print('AMG-RAG System successfully initialized!')


In [ ]:
system_b = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)
print("System B initialized.")


### 2.4 Labeled test set (identical to Part 1)

In [ ]:
test_reports = [
    # --- A. Genuine contradictions (same structure, opposite claims) ---
    {"report": "The heart size is within normal limits. Marked cardiomegaly is present, the heart is grossly enlarged. Lungs are clear bilaterally.", "expected_label": 1},
    {"report": "Frontal and lateral chest radiographs. There is no evidence of pleural effusion. A large left pleural effusion is identified. No pneumothorax is seen.", "expected_label": 1},
    {"report": "Both lung fields are clear without focal opacity. Extensive airspace consolidation is present in the left upper lobe, compatible with pneumonia. Heart size is normal.", "expected_label": 1},
    {"report": "No pneumothorax is identified. A moderate right-sided pneumothorax is seen at the apex. The cardiomediastinal silhouette is unremarkable.", "expected_label": 1},
    {"report": "The mediastinal contour is normal in width. There is marked mediastinal widening, raising concern for aortic injury. The lungs are clear.", "expected_label": 1},
    {"report": "No fracture is seen on this study. An acute, displaced fracture of the left 5th rib is identified. Lungs are clear bilaterally.", "expected_label": 1},
    {"report": "The trachea is midline without deviation. The trachea is deviated markedly to the right. Heart size is within normal limits.", "expected_label": 1},
    {"report": "No acute cardiopulmonary process is identified. There is acute pulmonary edema with diffuse bilateral airspace opacities. The heart is normal in size.", "expected_label": 1},
    {"report": "The osseous structures are intact without fracture. A pathologic compression fracture of the T8 vertebral body is noted. Lungs are clear.", "expected_label": 1},

    # --- B. Self-contradictory single statements ---
    {"report": "The lungs are clear bilaterally with diffuse bilateral ground-glass consolidation. Heart size is normal.", "expected_label": 1},
    {"report": "Heart size is normal in caliber; nonetheless, there is severe, massive cardiac enlargement. Lungs are otherwise unremarkable.", "expected_label": 1},
    {"report": "There is no evidence of pleural fluid; small-to-moderate bilateral pleural effusions are present. Lungs are otherwise clear.", "expected_label": 1},

    # --- C. Physiologically / anatomically impossible ---
    {"report": "The patient is noted to have four lobes within the right lung. Heart size is normal.", "expected_label": 1},
    {"report": "Both kidneys are surgically absent; there is, however, mild enlargement of the left kidney. Lungs are clear.", "expected_label": 1},
    {"report": "The heart demonstrates five distinct cardiac chambers on this study. Lungs are clear bilaterally.", "expected_label": 1},

    # --- D. Near-misses that should NOT be flagged ---
    {"report": "Heart size is normal. Tortuous and mildly prominent thoracic aorta, likely degenerative. Lungs are clear.", "expected_label": 0},
    {"report": "Lungs are clear of consolidation. Mild bibasilar atelectasis is present. Heart size is within normal limits.", "expected_label": 0},
    {"report": "No acute findings. A stable, densely calcified granuloma is again seen, unchanged from prior examination. Heart size is normal.", "expected_label": 0},
    {"report": "No focal consolidation to suggest pneumonia. A small incidental pulmonary nodule is noted; follow-up recommended. Lungs otherwise clear.", "expected_label": 0},
    {"report": "Heart size is normal. Mildly tortuous descending thoracic aorta, likely chronic and age-related. Lungs are clear.", "expected_label": 0},
    {"report": "No acute fracture identified. Old, well-healed fracture deformity of the right clavicle. Lungs are clear bilaterally.", "expected_label": 0},
    {"report": "Lungs are clear without consolidation. Mild chronic interstitial changes, stable in appearance. Heart size is normal.", "expected_label": 0},
    {"report": "No acute cardiopulmonary abnormality. Prominent but stable pulmonary vasculature, unchanged from prior studies. Heart size is normal.", "expected_label": 0},

    # --- E. Ambiguous / severity calibration cases ---
    {"report": "No acute findings. Chronic, longstanding prominence of the pulmonary arteries, stable compared to remote priors. Heart size is normal.", "expected_label": 0},
    {"report": "The heart is top-normal in size. Borderline cardiac enlargement is noted, unchanged from six months ago. Lungs are clear.", "expected_label": 0},
    {"report": "No acute findings. There is new, interval enlargement of the pulmonary trunk compared to the prior study performed one week ago. Lungs are clear.", "expected_label": 1},
]

print(f"Total test reports: {len(test_reports)}")
print(f"Positive (should flag): {sum(r['expected_label'] for r in test_reports)}")
print(f"Negative (should not flag): {sum(1 - r['expected_label'] for r in test_reports)}")


### 2.5 Run System B over the test set, timing each report

In [ ]:
import time
import io
import contextlib
from tqdm import tqdm

system_b_results = []

for i, item in enumerate(tqdm(test_reports, desc="System B")):
    report_text = item["report"]
    expected = item["expected_label"]

    start = time.perf_counter()
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            result = system_b.evaluate_medical_report(report_text)
        elapsed = time.perf_counter() - start
        error = None
        predicted = int(result["hallucination_detected"])
        confidence = result.get("overall_confidence", None)
    except Exception as e:
        elapsed = time.perf_counter() - start
        error = str(e)
        predicted = None
        confidence = None

    system_b_results.append({
        "report_id": i + 1,
        "report": report_text,
        "expected_label": expected,
        "predicted_label": predicted,
        "overall_confidence": confidence,
        "latency_sec": elapsed,
        "error": error,
    })

print(f"Done. {sum(1 for r in system_b_results if r['error'])} report(s) errored out.")


### 2.6 Save System B's results to disk

In [ ]:
import json

with open("system_b_results.json", "w") as f:
    json.dump(system_b_results, f, indent=2)

print("Saved system_b_results.json")
print("\nNow continue with Part 3 below (same session is fine, or a fresh restart --")
print("Part 3 only reads the two saved JSON files, it doesn't need either system's live objects).")


---
# Part 3: Side-by-Side Comparison

Loads both saved result files and computes Accuracy, Precision, Recall, F1, AUROC,
and latency for each system independently, then compares them directly. Requires
`system_a_results.json` and `system_b_results.json` to already exist (run Parts 1
and 2 first, in separate runtime sessions if needed).

In [ ]:
%pip install -q pandas scikit-learn matplotlib seaborn numpy


In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

with open("system_a_results.json") as f:
    system_a_results = json.load(f)
with open("system_b_results.json") as f:
    system_b_results = json.load(f)

df_a = pd.DataFrame(system_a_results)
df_b = pd.DataFrame(system_b_results)


def compute_metrics(df, name):
    valid = df[df["error"].isna()].copy()
    if len(valid) < len(df):
        print(f"[{name}] WARNING: {len(df) - len(valid)} report(s) errored and are excluded:")
        print(df[df["error"].notna()][["report_id", "error"]])

    y_true = valid["expected_label"].astype(int)
    y_pred = valid["predicted_label"].astype(int)

    metrics = {
        "system": name,
        "n_evaluated": len(valid),
        "n_errors": len(df) - len(valid),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mean_latency_sec": valid["latency_sec"].mean(),
        "median_latency_sec": valid["latency_sec"].median(),
        "p95_latency_sec": np.percentile(valid["latency_sec"], 95),
        "total_latency_sec": valid["latency_sec"].sum(),
    }

    # AUROC from the continuous confidence score (inverted: higher = more
    # likely hallucinated), same convention as the single-system eval notebook.
    try:
        hallucination_score = 1 - valid["overall_confidence"].astype(float)
        metrics["auroc"] = roc_auc_score(y_true, hallucination_score)
    except ValueError:
        metrics["auroc"] = float("nan")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    metrics["confusion_matrix"] = cm

    return metrics, valid


metrics_a, valid_a = compute_metrics(df_a, "System A (5-step, contradiction engine)")
metrics_b, valid_b = compute_metrics(df_b, "System B (single-verifier)")

comparison_df = pd.DataFrame([
    {k: v for k, v in metrics_a.items() if k != "confusion_matrix"},
    {k: v for k, v in metrics_b.items() if k != "confusion_matrix"},
]).set_index("system")

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
comparison_df


### 3.1 Confusion matrices, side by side

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metrics, title in zip(axes, [metrics_a, metrics_b], [metrics_a["system"], metrics_b["system"]]):
    sns.heatmap(metrics["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not hallucinated", "Hallucinated"],
                yticklabels=["Not hallucinated", "Hallucinated"], ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


### 3.2 Metric-by-metric bar chart

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1", "auroc"]
plot_df = comparison_df[metric_cols].reset_index().melt(id_vars="system", var_name="metric", value_name="score")

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="system")
plt.title("Classification Metrics: System A vs. System B")
plt.ylim(0, 1.05)
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


### 3.3 Latency comparison

In [ ]:
latency_cols = ["mean_latency_sec", "median_latency_sec", "p95_latency_sec"]
latency_plot_df = comparison_df[latency_cols].reset_index().melt(id_vars="system", var_name="metric", value_name="seconds")

plt.figure(figsize=(9, 5))
sns.barplot(data=latency_plot_df, x="metric", y="seconds", hue="system")
plt.title("Latency: System A vs. System B")
plt.tight_layout()
plt.show()

print(f"Total time to evaluate all {len(valid_a)} reports:")
print(f"  System A: {metrics_a['total_latency_sec']:.1f} s")
print(f"  System B: {metrics_b['total_latency_sec']:.1f} s")


### 3.4 Where they disagree

Reports where System A and System B reached different verdicts on the same input --
useful for understanding *why* one architecture outperforms the other, not just *that*
it does.

In [ ]:
merged = valid_a[["report_id", "report", "expected_label", "predicted_label", "overall_confidence"]].merge(
    valid_b[["report_id", "predicted_label", "overall_confidence"]],
    on="report_id", suffixes=("_A", "_B")
)

disagreements = merged[merged["predicted_label_A"] != merged["predicted_label_B"]]
print(f"Reports where System A and System B disagree: {len(disagreements)} / {len(merged)}")
disagreements[[
    "report_id", "report", "expected_label",
    "predicted_label_A", "overall_confidence_A",
    "predicted_label_B", "overall_confidence_B",
]]
